## 1. Apa itu activation function?

Activation function diterapkan pada neuron saat prediksi. Chapter ini membahas fungsi nonlinear untuk hidden layer dan output layer.

- **Sigmoid** memetakan nilai ke rentang 0–1.
- **Tanh** memetakan nilai ke rentang -1–1.
- **Softmax** mengubah skor beberapa kelas menjadi distribusi probabilitas yang jumlahnya 1.

Activation function yang berguna mendukung pembelajaran nonlinear dan cukup efisien untuk dihitung berulang kali.

## 2. Tanh dan turunannya

Pada contoh jaringan MNIST, tanh digunakan sebagai aktivasi hidden layer. Turunannya digunakan saat menghitung delta pada backpropagation.

In [1]:
def tanh(x):
    return np.tanh(x)

def tanh2deriv(output):
    return 1 - (output ** 2)


## 3. Softmax untuk output multi-kelas

Softmax mengubah skor output menjadi probabilitas untuk tiap kelas. Pada klasifikasi MNIST, ada 10 kelas digit (0–9), dan jumlah probabilitas dalam satu baris adalah 1.

In [2]:
def softmax(x):
    temp = np.exp(x)
    return temp / np.sum(temp, axis=1, keepdims=True)


### Contoh singkat

Contoh berikut mengilustrasikan perilaku softmax. Fungsi menerima array dua dimensi, dengan satu baris untuk setiap sampel.

In [3]:
import numpy as np

scores = np.array([[0.0, 0.0, 0.0, 100.0]])
probabilities = softmax(scores)

print(probabilities)
print(np.sum(probabilities, axis=1))


[[3.72007598e-44 3.72007598e-44 3.72007598e-44 1.00000000e+00]]
[1.]


## 4. Menyiapkan dataset MNIST

Contoh menggunakan 1.000 gambar training. Setiap gambar 28×28 diubah menjadi vektor 784 piksel dan dinormalisasi dengan membagi nilai piksel dengan 255. Label digit diubah menjadi one-hot encoding dengan 10 kolom.

Bagian ini menggunakan `keras.datasets.mnist`, seperti contoh buku.

In [5]:
import numpy as np, sys
np.random.seed(1)

from keras.datasets import mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

images, labels = (x_train[0:1000].reshape(1000,28*28) / 255, y_train[0:1000])
one_hot_labels = np.zeros((len(labels),10))
for i,l in enumerate(labels):
    one_hot_labels[i][l] = 1
labels = one_hot_labels

test_images = x_test.reshape(len(x_test),28*28) / 255
test_labels = np.zeros((len(y_test),10))
for i,l in enumerate(y_test):
    test_labels[i][l] = 1


## 5. Jaringan MNIST dengan tanh dan softmax

Contoh memakai tanh pada hidden layer dan softmax pada output layer. Bobot awal hidden layer dibuat lebih kecil, dan `alpha` disetel ulang.

In [7]:
alpha, iterations, hidden_size = (2, 300, 100)
pixels_per_image, num_labels = (784, 10)
batch_size = 100

weights_0_1 = 0.02*np.random.random((pixels_per_image,hidden_size))-0.01
weights_1_2 = 0.2*np.random.random((hidden_size,num_labels)) - 0.1

for j in range(iterations):
    correct_cnt = 0
    for i in range(int(len(images) / batch_size)):
        batch_start, batch_end = ((i * batch_size),((i+1)*batch_size))
        layer_0 = images[batch_start:batch_end]
        layer_1 = tanh(np.dot(layer_0,weights_0_1))
        dropout_mask = np.random.randint(2,size=layer_1.shape)
        layer_1 *= dropout_mask * 2
        layer_2 = softmax(np.dot(layer_1,weights_1_2))

        for k in range(batch_size):
            correct_cnt += int(np.argmax(layer_2[k:k+1]) == np.argmax(labels[batch_start+k:batch_start+k+1]))

        layer_2_delta = (labels[batch_start:batch_end]-layer_2) / (batch_size * layer_2.shape[0])
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * tanh2deriv(layer_1)
        layer_1_delta *= dropout_mask

        weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
        weights_0_1 += alpha * layer_0.T.dot(layer_1_delta)

    test_correct_cnt = 0
    for i in range(len(test_images)):
        layer_0 = test_images[i:i+1]
        layer_1 = tanh(np.dot(layer_0,weights_0_1))
        layer_2 = np.dot(layer_1,weights_1_2)
        test_correct_cnt += int(np.argmax(layer_2) == np.argmax(test_labels[i:i+1]))

    if(j % 10 == 0):
        sys.stdout.write("\n" + "I:" + str(j) +
                         " Test-Acc:" + str(test_correct_cnt/float(len(test_images))) +
                         " Train-Acc:" + str(correct_cnt/float(len(images))))



I:0 Test-Acc:0.3503 Train-Acc:0.181
I:10 Test-Acc:0.6734 Train-Acc:0.699
I:20 Test-Acc:0.7134 Train-Acc:0.725
I:30 Test-Acc:0.7473 Train-Acc:0.791
I:40 Test-Acc:0.7801 Train-Acc:0.817
I:50 Test-Acc:0.7971 Train-Acc:0.84
I:60 Test-Acc:0.8117 Train-Acc:0.847
I:70 Test-Acc:0.8199 Train-Acc:0.865
I:80 Test-Acc:0.8272 Train-Acc:0.87
I:90 Test-Acc:0.8347 Train-Acc:0.882
I:100 Test-Acc:0.8395 Train-Acc:0.886
I:110 Test-Acc:0.8425 Train-Acc:0.893
I:120 Test-Acc:0.8458 Train-Acc:0.902
I:130 Test-Acc:0.849 Train-Acc:0.907
I:140 Test-Acc:0.8522 Train-Acc:0.904
I:150 Test-Acc:0.8543 Train-Acc:0.916
I:160 Test-Acc:0.8566 Train-Acc:0.913
I:170 Test-Acc:0.8592 Train-Acc:0.919
I:180 Test-Acc:0.8604 Train-Acc:0.919
I:190 Test-Acc:0.8619 Train-Acc:0.929
I:200 Test-Acc:0.8634 Train-Acc:0.926
I:210 Test-Acc:0.8644 Train-Acc:0.928
I:220 Test-Acc:0.8647 Train-Acc:0.94
I:230 Test-Acc:0.8653 Train-Acc:0.936
I:240 Test-Acc:0.8656 Train-Acc:0.937
I:250 Test-Acc:0.868 Train-Acc:0.948
I:260 Test-Acc:0.8685 Train

## 6. Memahami alur training

1. Mengambil satu batch gambar.
2. Menghitung hidden layer dengan `tanh`.
3. Membuat dan menerapkan `dropout_mask`.
4. Menghasilkan probabilitas kelas dengan `softmax`.
5. Menghitung prediksi benar.
6. Menghitung `layer_2_delta`, lalu meneruskannya ke hidden layer dengan turunan tanh.
7. Memperbarui `weights_1_2` dan `weights_0_1`.
8. Mengukur akurasi test dan mencetak hasil setiap 10 iterasi.

Pada evaluasi test, menghitung skor output dengan `np.dot` tanpa menerapkan softmax. Untuk memilih kelas dengan `argmax`, hal ini tetap menghasilkan kelas dengan skor tertinggi yang sama.